<a href="https://colab.research.google.com/github/boraevinc/projectlabs/blob/main/sports.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sports dataset for the Polymarket project

This notebook builds `sports.csv`: the 500 highest-volume Polymarket Sports markets that closed between 1 July and 8 November 2026, one row per market, with the columns:

`slug, category, volume, close_date, price_7d, price_3d, price_1d, outcome`

It also saves:
- `sports_working.csv`: the same 500 rows plus a `sport` column
- `sports_report.txt`: how many markets were dropped at each filter

Everything downloaded is kept in the `raw` folder (`raw/sports_events.json` for the events, `raw/prices/<token>.json` for the price histories), so a second run, or a run after a crash, only fetches what is missing.

## Decision log

Filters and outcomes:

* Downloaded the 20,000 highest-volume closed events tagged Sports (the 200-page cap), asking the server only for events scheduled to end after 1 June 2026
* Dropped 2 events also tagged Crypto, Tech or Politics, because the plan gives those categories priority over Sports
* Dropped 507,700 markets that did not resolve cleanly Yes or No (cancelled, settled 50/50, or not a Yes/No question: many sports markets use team names as outcomes, so they have no "Yes" price)
* Dropped 41,014 markets that closed outside 1 July to 8 November 2026
* Dropped 87,634 markets with under $1,000 of volume: their price comes from quotes, not trading, so it says little about what people think
* A shared rule is to keep only events with a single market. Sports games are multi-market events (win / draw / win, spreads, totals), so that rule left only 17 usable markets. Instead, ranked individual Yes/No markets by volume and kept only the highest-volume market of each event, so a game is not counted two or three times (dropped 14,899 other markets)
* Took the outcome from the final prices
* Measured the 7, 3 and 1 day points back from the actual closing time (`closedTime`), not the scheduled end date, so the prices cannot leak the answer
* Used hourly prices, and took the last price at or before each time point, as long as it was at most 12 hours old
* Dropped 111 markets missing one of the three prices
* Kept the 500 highest-volume markets that were left
* Sport mix of the 500, reported afterwards and not designed: 463 football, 26 other, 7 basketball, 4 tennis

In [2]:
import json, os, time, requests, pandas as pd

## Settings

The two Polymarket APIs:
- Gamma holds events, markets, tags and final outcomes
- CLOB holds the price histories

In [3]:
GAMMA = "https://gamma-api.polymarket.com"
CLOB = "https://clob.polymarket.com"

The Sports tag (from the project plan)

In [4]:
SPORTS_TAG_ID = 1

Date range for the closing time (`closedTime`, UTC), both ends included

In [5]:
START = pd.Timestamp("2026-07-01", tz="UTC")
END = pd.Timestamp("2026-11-08 23:59:59", tz="UTC")

`TOTAL`: how many markets we want in the final table (the top `TOTAL` by volume)

In [6]:
TOTAL = 500

**Design choice.** A price only counts if it is at most this many hours older than the 7, 3 or 1 day time point, so a "7-day price" cannot be several days stale.

In [7]:
MAX_STALE_H = 12

How much to download:

- `MAX_PAGES`: safety cap on the event download (200 pages × 100 events = 20,000 events).
- `PREFILTER_BY_END_DATE`: ask the server only for events *scheduled* to end after 1 June 2026, so we do not page through every sports event ever. The exact filter is still `closedTime`.

In [8]:
MAX_PAGES = 200
PREFILTER_BY_END_DATE = True

**Design choice (deviation from the shared plan, tell the team).** Sports games are multi-market events, so we rank individual Yes/No *markets* by volume instead of requiring a one-market event.

- `ALLOW_MULTI_MARKET`: set to `False` to go back to the strict plan rule.
- `MAX_PER_EVENT`: keep only this many markets per event (the highest-volume ones). `1` avoids counting one game two or three times, since home win / draw / away win are dependent.
- `MIN_VOLUME`: ignore markets with (almost) no trading, because their prices are not real signals. Set to `0` to disable.

In [9]:
ALLOW_MULTI_MARKET = True
MIN_VOLUME = 1000
MAX_PER_EVENT = 1

Sport labels, matched against the event's tag slugs. The order is the priority if an event has several.

This only labels each market with a sport (the `sport` column in `sports_working.csv`). It does not affect which markets are selected. Anything not listed is labelled `other`, and the unmatched tag slugs are listed in the report, so the map can be extended after a run.

In [10]:
SPORT_MAP = {
    "football":   ["soccer", "epl", "la-liga", "serie-a", "bundesliga", "ligue-1",
                   "champions-league", "mls", "football"],
    "basketball": ["nba", "basketball", "wnba", "ncaab", "cbb"],
    "american_football": ["nfl", "cfb", "ncaaf", "college-football"],
    "tennis":     ["tennis", "atp", "wta"],
}

The folder for the cached raw downloads

In [11]:
os.makedirs("raw/prices", exist_ok=True)

## Small helpers

`get` makes one web request and returns the reply as Python lists and dictionaries. Timeouts, rate limits (status 429) and server errors (500 and above) wait and try again (1, 2, 4... seconds) instead of crashing the run. After 6 failed attempts it stops with an error.

In [12]:
def get(url, **params):
    for i in range(6):
        try:
            r = requests.get(url, params=params, timeout=60)
        except requests.RequestException:
            time.sleep(2 ** i); continue
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(2 ** i); continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"gave up on {url}")

The fields `outcomes`, `outcomePrices` and `clobTokenIds` arrive as text that looks like a list, e.g. `'["Yes", "No"]'`

`jl` turns such text into a real list.

In [13]:
def jl(x):
    return json.loads(x) if isinstance(x, str) else x

## Step 1: download the events

`download_events` pages through the closed Sports events, highest volume first (100 per page, following the cursor the API returns), up to `MAX_PAGES` pages.

The result is saved to `raw/sports_events.json`. If that file exists it is reused, not downloaded again.

In [14]:
def download_events():
    path = "raw/sports_events.json"
    if os.path.exists(path):
        return json.load(open(path))
    events, cursor = [], None
    for _ in range(MAX_PAGES):
        p = dict(tag_id=SPORTS_TAG_ID, closed="true", order="volume",
                 ascending="false", limit=100)
        if PREFILTER_BY_END_DATE:
            p["end_date_min"] = "2026-06-01T00:00:00Z"
        if cursor:
            p["after_cursor"] = cursor
        d = get(f"{GAMMA}/events/keyset", **p)
        batch = d["events"]
        events += batch
        cursor = d.get("next_cursor")
        if not batch or not cursor:
            break
    json.dump(events, open(path, "w"))
    return events

## Step 2: build the market list

`sport_of` labels an event with a sport from its tag slugs (the first match in `SPORT_MAP` wins).

Tags that match nothing are counted in `unmapped`, so they show up in the report, and the event is labelled `other`.

In [15]:
def sport_of(ev, unmapped):
    slugs = [t.get("slug", "") for t in ev.get("tags", [])]
    for sport, keys in SPORT_MAP.items():
        if any(s in keys for s in slugs):
            return sport
    for s in slugs:
        unmapped[s] = unmapped.get(s, 0) + 1
    return "other"

`build_candidates` turns the events into one row per market (slug, sport, volume, closing time, Yes token ID, outcome) and applies the filters from the decision log, in this order:

1. event also tagged Crypto, Tech or Politics → skipped (the plan's priority order)
2. outcomes must include "Yes" and the final prices must be exactly 1 and 0 (the outcome is 1 if Yes won, 0 if No won)
3. `closedTime` must fall inside `START` to `END`
4. volume must be at least `MIN_VOLUME`
5. keep only the highest-volume market(s) of each event (`MAX_PER_EVENT`)

Every dropped market is counted in `rep` (written to `sports_report.txt`). The result is sorted from highest volume to lowest.

In [16]:
def build_candidates(events, rep):
    rows, unmapped = [], {}
    rep["events_total"] = len(events)
    rep["dropped_multi_market"] = rep["dropped_not_resolved_yes_no"] = 0
    rep["dropped_outside_window"] = 0
    rep["dropped_higher_priority_category"] = 0
    rep["dropped_extra_markets_per_event"] = 0
    rep["dropped_below_min_volume"] = 0
    HIGHER = {"21", "1401", "2"}
    for ev in events:
        if {str(t.get("id")) for t in ev.get("tags", [])} & HIGHER:
            rep["dropped_higher_priority_category"] += 1; continue
        mk = ev.get("markets", [])
        if len(mk) > 1 and not ALLOW_MULTI_MARKET:
            rep["dropped_multi_market"] += len(mk); continue
        sport = sport_of(ev, unmapped)
        ev_rows = []
        for m in mk:
            try:
                prices = [float(x) for x in jl(m["outcomePrices"])]
                outs = jl(m["outcomes"]); toks = jl(m["clobTokenIds"])
            except Exception:
                rep["dropped_not_resolved_yes_no"] += 1; continue
            if "Yes" not in outs or sorted(prices) != [0.0, 1.0]:
                rep["dropped_not_resolved_yes_no"] += 1; continue
            iy = outs.index("Yes")
            closed = pd.to_datetime(m.get("closedTime") or ev.get("closedTime"),
                                    utc=True, errors="coerce")
            if pd.isna(closed) or not (START <= closed <= END):
                rep["dropped_outside_window"] += 1; continue
            vol = float(m.get("volume") or m.get("volumeNum") or 0)
            if vol < MIN_VOLUME:
                rep["dropped_below_min_volume"] += 1; continue
            ev_rows.append(dict(slug=m["slug"], sport=sport,
                                volume=vol,
                                closed=closed, token=toks[iy],
                                outcome=int(prices[iy] == 1.0)))
        ev_rows.sort(key=lambda r: -r["volume"])
        rep["dropped_extra_markets_per_event"] += max(0, len(ev_rows) - MAX_PER_EVENT)
        rows += ev_rows[:MAX_PER_EVENT]
    rep["unmapped_tag_slugs"] = dict(sorted(unmapped.items(), key=lambda kv: -kv[1])[:30])
    return pd.DataFrame(rows).sort_values("volume", ascending=False)

## Step 3: download the price histories

`price_history` gets the hourly Yes prices (`fidelity` 60 = one point per 60 minutes) for the 8 days before the close

Each point is `{"t": seconds since 1970, "p": price}`. The history is saved to `raw/prices/<token>.json` and only downloaded if that file is missing.

In [17]:
def price_history(token, closed):
    path = f"raw/prices/{token}.json"
    if os.path.exists(path):
        return json.load(open(path))
    start = int((closed - pd.Timedelta(days=8)).timestamp())
    end = int(closed.timestamp())
    d = get(f"{CLOB}/prices-history", market=token, startTs=start,
            endTs=end, fidelity=60)
    h = d.get("history", [])
    json.dump(h, open(path, "w"))
    return h

`price_at` finds the last price at or before time `ts`

It returns `None` if there is no price there, or if the last one is more than `MAX_STALE_H` hours older than `ts`

In [18]:
def price_at(hist, ts):
    best = None
    for pt in hist:
        if pt["t"] <= ts:
            best = pt
    if best is None or ts - best["t"] > MAX_STALE_H * 3600:
        return None
    return float(best["p"])

- `FETCH_ERRORS`: markets whose download kept failing. They are skipped and not cached, so running the notebook again retries them.
- `SEEN`: running count used for the progress message.

In [19]:
FETCH_ERRORS = []
SEEN = [0]

`try_market` gets the 7, 3 and 1 day prices for one candidate and prints progress every 50 markets

It returns a finished row, or `None` if the download failed or any of the three prices is missing (a market needs all three to be kept)

In [20]:
def try_market(r):
    try:
        h = price_history(r.token, r.closed)
    except Exception:
        FETCH_ERRORS.append(r.slug)
        return None
    SEEN[0] += 1
    if SEEN[0] % 50 == 0:
        print(f"  ...checked {SEEN[0]} markets", flush=True)
    c = int(r.closed.timestamp())
    ps = [price_at(h, c - d * 86400) for d in (7, 3, 1)]
    if any(p is None for p in ps):
        return None
    return dict(slug=r.slug, category="Sports", volume=r.volume,
                close_date=r.closed.strftime("%Y-%m-%d"),
                price_7d=ps[0], price_3d=ps[1], price_1d=ps[2],
                outcome=r.outcome, sport=r.sport)

## Step 4: select the 500 and save

`main` runs the steps in order:

1. Builds the candidate list from the downloaded events.
2. Goes down the candidates from highest volume and keeps those with all three prices until it has `TOTAL` (500) rows.
3. Checks the table (no repeated slug, no blank cells), then saves `sports_working.csv` (with the `sport` column), `sports.csv` (exactly the shared columns) and `sports_report.txt`, and prints the report

In [21]:
def main():
    rep = {}
    cand = build_candidates(download_events(), rep)
    rep["candidates_by_sport"] = cand.sport.value_counts().to_dict()
    kept, no_price = [], 0
    for r in cand.itertuples():
        if len(kept) >= TOTAL: break
        row = try_market(r)
        if row: kept.append(row)
        else: no_price += 1
    rep["dropped_missing_price_point"] = no_price - len(FETCH_ERRORS)
    rep["failed_downloads_rerun_to_retry"] = len(FETCH_ERRORS)
    df = pd.DataFrame(kept).sort_values("volume", ascending=False)
    rep["final_rows"] = len(df)
    rep["final_by_sport"] = df.sport.value_counts().to_dict()
    df.to_csv("sports_working.csv", index=False)
    cols = ["slug", "category", "volume", "close_date",
            "price_7d", "price_3d", "price_1d", "outcome"]
    assert df.slug.is_unique and not df[cols].isna().any().any()
    df[cols].to_csv("sports.csv", index=False)
    open("sports_report.txt", "w").write(json.dumps(rep, indent=2, default=str))
    print(json.dumps(rep, indent=2, default=str))

Run main

In [22]:
if __name__ == "__main__":
    main()

  ...checked 50 markets
  ...checked 100 markets
  ...checked 150 markets
  ...checked 200 markets
  ...checked 250 markets
  ...checked 300 markets
  ...checked 350 markets
  ...checked 400 markets
  ...checked 450 markets
  ...checked 500 markets
  ...checked 550 markets
  ...checked 600 markets
{
  "events_total": 20000,
  "dropped_multi_market": 0,
  "dropped_not_resolved_yes_no": 515985,
  "dropped_outside_window": 43945,
  "dropped_higher_priority_category": 2,
  "dropped_extra_markets_per_event": 14676,
  "dropped_below_min_volume": 87410,
  "unmapped_tag_slugs": {
    "sports": 6574,
    "games": 6459,
    "esports": 3920,
    "baseball": 1710,
    "mlb": 1707,
    "counter-strike-2": 1686,
    "league-of-legends": 1017,
    "dota-2": 873,
    "valorant": 339,
    "hockey": 312,
    "nhl": 312,
    "ufc": 283,
    "cricket": 259,
    "international-cricket": 152,
    "lol-worlds-2026": 50,
    "mlb-playoffs": 40,
    "swiss-stage": 33,
    "blast": 32,
    "formula1": 31,
    "

## Result

In [23]:
pd.read_csv("sports.csv")

,slug,category,volume,close_date,price_7d,price_3d,price_1d,outcome
0,will-argentina-win-the-2026-fifa-world-cup-245,Sports,1.743157e+08,2026-07-20,0.1815,0.4155,0.4095,0
1,will-trump-be-in-the-wc-champions-photo-202606...,Sports,2.699014e+07,2026-07-24,0.7050,0.9960,0.9985,1
2,will-harry-kane-win-the-2026-ballon-dor,Sports,1.523629e+07,2026-10-27,0.7250,0.7655,0.7085,1
3,unl-nor-den-2026-09-24-nor,Sports,1.352195e+07,2026-09-24,0.5150,0.5550,0.5550,1
4,will-oliver-bearman-be-the-2026-f1-drivers-cha...,Sports,1.289782e+07,2026-09-13,0.0005,0.0005,0.0005,0
...,...,...,...,...,...,...,...,...
495,sea-par-tor-2026-10-19-tor,Sports,5.142179e+05,2026-10-19,0.3850,0.4050,0.4050,1
496,epl-cov-ful-2026-10-24-ful,Sports,5.129664e+05,2026-10-24,0.4050,0.4050,0.4250,1
497,lal-elc-bet-2026-11-08-bet,Sports,5.105840e+05,2026-11-08,0.4500,0.4650,0.4850,1
498,mex-tol-nec-2026-08-02-tol,Sports,5.079237e+05,2026-08-03,0.6800,0.6450,0.6250,1
